In [19]:
import numpy as np
import pandas as pd
import math, time
from datetime import datetime
from bs4 import BeautifulSoup
import re


In [11]:
xl = pd.ExcelFile("data/World Cup 2026.xlsx")
sheet_list = xl.sheet_names
print(sheet_list)

['Squads', 'Players', 'Squad Members']


In [12]:
# Import data
squads = pd.read_excel('data/World Cup 2026.xlsx', sheet_name="Squads")
players = pd.read_excel('data/World Cup 2026.xlsx', sheet_name="Players")
squad_members = pd.read_excel('data/World Cup 2026.xlsx', sheet_name="Squad Members")

/home/roman/Code/AIEngineering/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [6]:
squad_members.head()

,squad_id,player_id
0,1,WC-2026-001
1,1,WC-2026-002
2,1,WC-2026-003
3,1,WC-2026-004
4,1,WC-2026-005


In [8]:
squads.head()

,ID,Country,Details
0,1,Mexico,Squad for game against South Africa
1,2,South Africa,Squad for game agains Mexico


In [14]:
# Functions to scrape data from wikipedia
import requests
from requests.adapters import HTTPAdapter
from urllib3 import Retry
import random

class ScrapingException(BaseException):
    """Errors gotten when trying to scrape from wikipedia"""

# Use different user agents to mimic different browsers
USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
]

def get_session():
    """
    Function to return a session that I can use to make requests to wikipedia's pages
    """
    session = requests.Session()
    
    # Add retry logic
    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter=adapter)
    return session
    
SESSION = get_session()

def get_wikipedia_page(url):
    """
    A function to try getting a wikipedia page
    """
    headers = {"User-Agent": random.choice(USER_AGENTS)}
    # Make a request to Wikipedia
    try:
        print("Getting response")
        wikipedia_response = SESSION.get(url, headers=headers, timeout=15)
        if wikipedia_response.status_code != 200:
            raise ScrapingException(f"Wikipedia refused to send page with status code {wikipedia_response.status_code}")
        else:
            return wikipedia_response
    except Exception as e:
        print(f"Could not get page because of {e}")
        raise Exception("Unknown error")
    
jean_paul_response = get_wikipedia_page("https://en.wikipedia.org/wiki/Jean-Paul_Abalo")

Getting response


In [15]:
class PlayerDetails:
    def __init__(self, from_year: int | None, to_year: int | None, appearances: int | None, goals: int | None, team: str | None):
        self.from_year = from_year
        self.to_year = to_year
        self.appearances = appearances
        self.goals = goals
        self.team = team
        
    def __repr__(self):
        return f"Found data row with from: {self.from_year} to: {self.to_year} team: {self.team} appearances: {self.appearances} and goals: {self.goals}"

def extract_infobox(player_wikipedia: str, player_id: str, problematic_players: list[str]) -> list[PlayerDetails] | None:
    """
    Extract player details from a player's wikipedia page
    
    Args:
        player_wikipedia: URL of a player's wikipedia page
        player_id: id of player being processed
        problematic_players: list of players whose wikipedia page couldn't be parsed
        
    Returns:
        players: A list of player objects if response of player was parsable
    """
    try:    
        player_response = get_wikipedia_page(player_wikipedia)
        assert player_response.status_code == 200, "Can't parse an unsucessful response"
        
        # Parse HTML of the player's Wiki page
        soup = BeautifulSoup(player_response.text, 'html.parser')

        # Look for the players info box
        tables = soup.select_one("table.infobox")

        # Store unparsable player for later processing
        if not tables:
            print(f"Player does not have info box in wiki: {player_wikipedia}")
            problematic_players.append(player_id)
            return None

        player_details = []
        table_rows = tables.find_all("tr")
        current = 0
        section_name = ""
        content_num = 0

        # Scanning table rows
        while True:
            if current >= len(table_rows):
                print("Consumed all rows in table")
                break
            
            current_row = table_rows[current]
            
            # Checking if row is the heading of a section
            if len(current_row.find_all("td")) == 0:
                section_name = current_row.select_one("th").text
                content_num = 0
                
                # Check if it's the section we want
                if "Senior career" in section_name:
                    # While next item is content
                    while current + 1 < len(table_rows) and len(table_rows[current + 1].find_all("td")) > 0:
                        next_content = table_rows[current + 1]
                        # Check if it is a table row
                        if len(next_content.find_all("b")) > 0:
                            print(f"Found title row {next_content.find("b")}")
                        else:
                            # Work with data that looks like 1 th 3 td
                            data_timeline = next_content.find("th")
                            
                            from_year = None
                            to_year = None
                            team = None
                            appearances = None
                            goals = None
                            
                            # Extract from and to years
                            if data_timeline:
                                raw_timeline = data_timeline.get_text(strip=True)
                                if len(raw_timeline) > 4:
                                    from_year = int(re.sub(r'\D', '', raw_timeline[0:4]))
                                    if len(raw_timeline) > 5:
                                        to_year = int(re.sub(r'\D', '', raw_timeline[4:]))
                                else:
                                    from_year = int(re.sub(r'\D', '', raw_timeline))
                            
                            rest_data = next_content.find_all("td")
                            if len(rest_data) < 3:
                                print(f"Unkown rest data {rest_data}")
                                problematic_players.append(player_id)
                            # Get team, appearance, goals
                            else:
                                raw_appearance = rest_data[1].get_text(strip=True)
                                raw_goals = rest_data[2].get_text(strip=True)
                                team = rest_data[0].get_text(strip=True)
                                if len(raw_appearance) > 0 and raw_appearance != "?":
                                    appearances = int(re.sub(r'\D', '', raw_appearance))
                                if len(raw_goals) > 0 and raw_goals != "(?)":
                                    goals = int(re.sub(r'\D', '', raw_goals))
                        
                            print(f"Found data row with from: {from_year} to: {to_year} team: {team} appearances: {appearances} and goals: {goals}")
                            player_details.append(PlayerDetails(
                                from_year=from_year,
                                to_year=to_year,
                                appearances=appearances,
                                goals=goals,
                                team = team
                            ))
                            content_num += 1
                        
                        current += 1
                        
                    # Increment
                    current += 1
                    print(f"Section {section_name} had {content_num} items")
                else:
                    current += 1
            else:
                current += 1
                
        return player_details
    except Exception as e:
        print(f"Unexpected error {e}")
        problematic_players.append(player_id)
    except ScrapingException as e:
        print(f"Error getting Wikipedia page {e}")
        problematic_players.append(player_id)
        
problematic_players = []
player_info = extract_infobox("https://en.wikipedia.org/wiki/Jean-Paul_Abalo", 'P001', problematic_players)
print(player_info)

Getting response
Found title row <b>Team</b>
Found data row with from: 1992 to: 1993 team: OC Agaza appearances: None and goals: None
Found data row with from: 1993 to: 1995 team: Saint-Christophe Châteauroux appearances: 29 and goals: 1
Found data row with from: 1995 to: 2005 team: Amiens SC appearances: 273 and goals: 7
Found data row with from: 2005 to: None team: USL Dunkerque appearances: 4 and goals: 0
Found data row with from: 2006 to: None team: APOEL appearances: 3 and goals: 0
Found data row with from: 2006 to: None team: Ethnikos Piraeus appearances: 9 and goals: 0
Found data row with from: 2007 to: 2008 team: Al-Merrikh appearances: None and goals: None
Found data row with from: 2008 to: 2009 team: FC Déols 36 appearances: None and goals: None
Section Senior career* had 8 items
Consumed all rows in table
[Found data row with from: 1992 to: 1993 team: OC Agaza appearances: None and goals: None, Found data row with from: 1993 to: 1995 team: Saint-Christophe Châteauroux appear

In [16]:
class Player:
    def __init__(self, id: str, wikipedia_url: str, problematic_players: list[str]):
        self.id = id
        self.wikipedia_url = wikipedia_url
        self.wikipedia_details = self._extract_wikipedia_details(problematic_players=problematic_players)
        
    def _extract_wikipedia_details(self, problematic_players: list[str]) -> list[PlayerDetails] | None:
        """
        Returns player details that have been extracted from their wikipedia page
        
        Args:
            problematic_players : A list used to store all player ids that had an issue extracting details from
            
        Returns:
            player_details: A list of player details from wikipedia
        """
        return extract_infobox(self.wikipedia_url, self.id, problematic_players)
    
    def to_csv(self) -> list[str]:
        """ 
        Returns strings that will represent the player in a CSV file
        
        Returns:
            entries (str): Entries in CSV file
        """
        entries = []
        if self.wikipedia_details:
            for detail in self.wikipedia_details:
                entries.append(f"{self.id},{detail.appearances},{detail.team},{detail.goals},{detail.from_year},{detail.to_year}\n")
            
        return entries
    
    def __repr__(self):
        return f"Player: {self.wikipedia_details}"

In [7]:
players.head()

,ID,birth_date,goalkeeper,defender,midfielder,forward,count_tournaments,list_tournaments,wikipedia
0,WC-2026-001,1985-07-13 00:00:00,1,0,0,0,7,"2004,2006,2010,2014,2018,2022,2026",https://en.wikipedia.org/wiki/Guillermo_Ochoa
1,WC-2026-002,1994-08-15 00:00:00,0,1,0,0,3,"2018,2022,2026",https://en.wikipedia.org/wiki/Jes%C3%BAs_Gallardo
2,WC-2026-003,1998-10-22 00:00:00,0,1,0,0,2,"2022,2026",https://en.wikipedia.org/wiki/Johan_V%C3%A1squ...
3,WC-2026-004,1997-02-24 00:00:00,0,1,0,0,3,"2018,2022,2026",https://en.wikipedia.org/wiki/C%C3%A9sar_Montes
4,WC-2026-005,2000-05-23 00:00:00,0,0,1,0,2,"2022,2026",https://en.wikipedia.org/wiki/Israel_Reyes


## Extracting player statistics

In [21]:
batch_size = 100
total_batches = math.ceil(players.shape[0] / batch_size)
print(f"Total batches: {total_batches}")
extracted_players = []
problematic_players = []

file_name = f"players_{datetime.now()}.csv"
    
with open(file_name, 'a') as file:
    # Insert csv file header
    file.write("id,appearances,team,goals,from,to\n")
    
    # Get players
    for batch_num in range(total_batches):
        # Getting indexes of batch
        print(f"Processing batch #{batch_num}")
        start = batch_num * batch_size
        if batch_num == total_batches - 1:
            end = players.shape[0]
        else:
            end = start + batch_size
           
        # Storing details of players in batch 
        df_segment = players.iloc[start:end]
        for row in df_segment.itertuples():                  
            player = Player(
                id=row.ID,
                wikipedia_url=row.wikipedia,
                problematic_players=problematic_players
            )
            
            extracted_players.append(player)
            # Insert entries in CSV
            for entry in player.to_csv():
                file.write(entry)
            
        time.sleep(10)
    

Total batches: 1
Processing batch #0
Getting response
Found title row <b>Team</b>
Found data row with from: 2003 to: None team: Tigrillos Coapa[4][5] appearances: 12 and goals: 0
Found data row with from: 2004 to: 2011 team: América appearances: 239 and goals: 0
Found data row with from: 2004 to: None team: →San Luis(loan) appearances: 1 and goals: 0
Found data row with from: 2011 to: 2014 team: Ajaccio appearances: 112 and goals: 0
Found data row with from: 2014 to: 2017 team: Málaga appearances: 11 and goals: 0
Found data row with from: 2016 to: 2017 team: →Granada(loan) appearances: 38 and goals: 0
Found data row with from: 2017 to: 2019 team: Standard Liège appearances: 78 and goals: 0
Found data row with from: 2019 to: 2022 team: América appearances: 118 and goals: 0
Found data row with from: 2022 to: 2024 team: Salernitana appearances: 41 and goals: 0
Found data row with from: 2024 to: 2025 team: AVS appearances: 23 and goals: 0
Found data row with from: 2025 to: None team: AEL L

## Generating dataset


In [22]:
statistics = pd.read_csv("players_2026-06-11 21:42:43.026827.csv")
statistics.head()

,id,appearances,team,goals,from,to
0,WC-2026-001,12,Tigrillos Coapa[4][5],0,2003,NaN
1,WC-2026-001,239,América,0,2004,2011.0
2,WC-2026-001,1,→San Luis(loan),0,2004,NaN
3,WC-2026-001,112,Ajaccio,0,2011,2014.0
4,WC-2026-001,11,Málaga,0,2014,2017.0


In [41]:
def get_players_statistics(player_id: str, match_date: str):
    """ 
    Return statistics for a player:
        1. age
        2. forward
        3. middlefielder
        4. defense
        5. goalkeeper
        6. num_tournaments
        7. appearances
        8. goals
        9. average time per team
    
    Args:
        player_id (str): id of player
        match_date (str): day match was played
        
    Returns:
        x_i (ndarray): (1,9) array with statistics of each player
    """    
    player = players[players['ID'] == player_id]
    if len(player) < 1:
        return np.array([
            [0,0,0,0,0,0,0,0,0]
        ])
    
    # Age of player
    # raw_birthdate = player['birth_date'].iloc[0]
    birthdate = player['birth_date'].iloc[0]
    if (type(birthdate) == str):
        birthdate = datetime.strptime(birthdate.strip(), "%Y-%m-%d")
    tournament_date = datetime.strptime(match_date, "%Y-%m-%d")
    match_year = tournament_date.year
    difference_in_days = (tournament_date - birthdate).days
    age = int(difference_in_days / 365.25)
    
    forward = player['forward'].iloc[0]
    middlefielder = player['midfielder'].iloc[0]
    defense = player['defender'].iloc[0]
    goalkeeper = player['goalkeeper'].iloc[0]
    
    raw_tournaments = player['list_tournaments'].iloc[0].split(",")
    tournaments = [int(t) < match_year for t in raw_tournaments]
    num_tournaments = sum(tournaments)
    
    appearances = 0
    goals = 0
    average_time_per_team = 0
    num_teams = 0
    player_statistics = statistics[statistics['id'] == player_id]
    
    if len(player_statistics) < 1:
        return np.array(
            [[age, forward, middlefielder, defense, goalkeeper, num_tournaments, appearances, goals, average_time_per_team]]
        )
    
    # Get stats as per tournament year
    for stat in player_statistics.iterrows():
        s_from = stat[1]["from"]
        s_to = match_year if np.isnan(stat[1]["to"]) else stat[1]["to"]
        s_appearance = 0 if np.isnan(stat[1]['appearances']) else stat[1]['appearances']
        s_goals = 0 if np.isnan(stat[1]['goals']) else stat[1]['goals']
        
        
        if s_from < match_year and s_to < match_year:
            appearances += s_appearance
            goals += s_goals
            num_teams += 1
            average_time_per_team += s_to - s_from
        elif s_from < match_year:
            total = s_to - s_from
            num_years = match_year - s_from
            appearances += int(s_appearance * (num_years / total))
            goals += int(s_goals * (num_years / total))
            num_teams += 1
            average_time_per_team += num_years
            
            
    if num_teams == 0:
        average_time_per_team = 0
    else:
        average_time_per_team /= num_teams
        
    return np.array([
            [age, forward, middlefielder, defense, goalkeeper, num_tournaments, appearances, goals, average_time_per_team]
    ]
        )
    
player = get_players_statistics("WC-2026-001", "2026-06-11")
print(player.shape)

(1, 9)


In [42]:
def get_world_cup_match_training_features(home_squad: int, away_squad: int, match_date: str):
    """ 
    Function to get training example from a world cup match.
    
    Args:
        home_squad (scalar): id of home squad
        away_squad (scalar): id of away squad
        match_date (scalar): date of match
        
    Returns:
        x_i: (ndarray): A (22,9) ndarray that has the statistics of each player in home and away team
    """
    x_i = np.empty((0,9))
    
    year = 2026
    
    # Get players in home team (Assumes that first squad is first 11 in dataset for a particular team)
    # home_team = reduced_squad[(reduced_squad['year'] == year) & (reduced_squad['team_id'] == home_team_id)].head(n=11)
    home_team = squad_members[squad_members["squad_id"] == home_squad].head(n=11)
    home_team_player_ids = home_team['player_id']
    if len(home_team_player_ids) != 11:
        raise Exception("Could not get players for home team")
    
    player_ids = home_team_player_ids.tolist()

    # Get players in away team
    # away_team = reduced_squad[(reduced_squad['year'] == year) & (reduced_squad['team_id'] == away_team_id)].head(n=11)
    away_team = squad_members[squad_members["squad_id"] == away_squad].head(n=11)
    away_team_player_ids = away_team['player_id']
    if len(away_team_player_ids) != 11:
        raise Exception("Could not get away players")
    
    player_ids.extend(away_team_player_ids.tolist())
    
    # For each player
    for player_id in player_ids:
        # Get statistics
        stats = get_players_statistics(player_id=player_id, match_date=match_date)
        # Store in x_i
        x_i = np.append(x_i, stats, axis=0)
    
    return x_i

X_pred = get_world_cup_match_training_features(home_squad=1, away_squad=2, match_date="2026-06-11")
np.save("WorldCup2026.npy", X_pred)